# 🔬 MammoAI — Detección de Cáncer de Mama

> **Sistema Open Source** de IA para detección, localización y análisis cuantitativo de mamografías.
> Arquitecturas: EfficientNet · ConvNeXt · ViT · Faster R-CNN · DETR
> Dataset: CBIS-DDSM

---

## 📋 Instrucciones previas

Antes de ejecutar el notebook, configura tus secrets en Colab:

1. En el menú de la izquierda, haz clic en el ícono 🔑 **(Secrets)**
2. Agrega los siguientes secrets:
   - **`GITHUB_TOKEN`** → Tu Personal Access Token de GitHub (con permisos `repo`)
   - **`HF_TOKEN`** → (Opcional) Tu token de HuggingFace para descargar modelos privados

> ⚠️ **NUNCA** escribas tu token directamente en el código.

---

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 1 — Verificación de GPU y entorno                   ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, os

# Verificar GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU disponible:')
    lines = result.stdout.split('\n')
    for line in lines[8:12]:
        print(' ', line)
else:
    print('⚠️  No se detectó GPU. Ve a: Entorno de ejecución → Cambiar tipo de entorno → GPU')

# Verificar Python y CUDA
import torch
print(f'\n🐍 Python {sys.version.split()[0]}')
print(f'🔥 PyTorch {torch.__version__}')
print(f'💻 CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🎮 GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive montado.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 2 — Instalación de dependencias                     ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys
print('Instalando dependencias...')
print('   (Este proceso tarda ~5-8 minutos en la primera ejecucion)')

deps = [
    'torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118',
    'gradio plotly>=5.18.0',
    'timm>=0.9.12 opencv-python-headless pillow scikit-image',
    'grad-cam>=1.4.8',
    'ultralytics>=8.0.0',
    'pydicom>=2.4.3',
    'huggingface_hub>=0.20.3 datasets>=2.16.0 transformers>=4.37.0',
    'peft>=0.9.0 accelerate>=0.26.0 bitsandbytes>=0.43.0',
    'tensorflow-cpu tensorflow-datasets>=4.9.0',
    'scikit-learn numpy',
    'onnx safetensors reportlab',
    'gitpython tqdm requests',
]

for dep in deps:
    pkg = dep.split()[0].split('>=')[0].split('==')[0]
    print(f'  Installing: {pkg}...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + dep.split(),
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'  WARNING: {result.stderr[:200]}')

print('\nTodas las dependencias instaladas ✅')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 3 — Configuración de credenciales (Colab Secrets)   ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import userdata

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print('✅ GITHUB_TOKEN cargado desde Colab Secrets.')
except Exception:
    GITHUB_TOKEN = ''
    print('⚠️  GITHUB_TOKEN no encontrado en Secrets.')
    print('   Ve a 🔑 Secrets en el panel izquierdo y agrega GITHUB_TOKEN.')

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('✅ HF_TOKEN configurado.')
except Exception:
    HF_TOKEN = ''
    print('ℹ️  HF_TOKEN no configurado (opcional para modelos públicos).')

print(f'\n📋 Configuración:')
print(f"   GitHub Token: {'Configurado ✅' if GITHUB_TOKEN else 'No configurado ⚠️'}")
print(f"   HF Token:     {'Configurado ✅' if HF_TOKEN else 'No configurado (OK para modelos públicos)'}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 4 — Clonar / Actualizar repositorio GitHub          ║
# ╚══════════════════════════════════════════════════════════════╝
import os, subprocess, sys
from pathlib import Path

REPO_URL   = 'https://github.com/AderDevP/IA-Models'
LOCAL_DIR  = '/content/IA-Models'
BRANCH     = 'main'

if GITHUB_TOKEN:
    auth_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
else:
    auth_url = REPO_URL

if Path(LOCAL_DIR).exists():
    print(f'📁 Repositorio ya existe. Forzando sincronización con origin/{BRANCH}...')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=LOCAL_DIR, capture_output=True)
    result = subprocess.run(
        ['git', 'reset', '--hard', f'origin/{BRANCH}'],
        cwd=LOCAL_DIR, capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    print(f'⬇️  Clonando {REPO_URL}...')
    result = subprocess.run(
        ['git', 'clone', '--branch', BRANCH, auth_url, LOCAL_DIR],
        capture_output=True, text=True
    )
    safe_output = (result.stdout + result.stderr).replace(GITHUB_TOKEN or 'x', '***')
    print(safe_output)

if LOCAL_DIR not in sys.path:
    sys.path.insert(0, LOCAL_DIR)

os.chdir(LOCAL_DIR)
print(f'\n✅ Directorio de trabajo: {os.getcwd()}')
print(f'📂 Archivos del proyecto:')
for f in sorted(Path(LOCAL_DIR).iterdir()):
    icon = '📁' if f.is_dir() else '📄'
    print(f'   {icon} {f.name}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 5 — Descarga automatica de modelos preentrenados     ║
# ╚══════════════════════════════════════════════════════════════╝
import importlib, logging, sys

# Logging live
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(name)s - %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
print('[LOG] Logging activado - veras en tiempo real todo lo que hace el dashboard.')

try:
    if HF_TOKEN:
        import huggingface_hub
        huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
        print('[OK] HuggingFace login OK (MedGemma disponible)')
except Exception as e:
    print(f'[INFO] HF login omitido: {e}')

import model_downloader
importlib.reload(model_downloader)
from model_downloader import ModelDownloader

downloader = ModelDownloader()

print('\nEstado actual de modelos:')
print(downloader.status_report())

AUTO_DOWNLOAD_MODELS = [
    'medgemma_cbis_ddsm',
    'efficientnet_b4_cbis',
]

print('\nDescargando modelos seleccionados...')
for model_id in AUTO_DOWNLOAD_MODELS:
    if not downloader.is_installed(model_id):
        print(f'\nDescargando: {model_id}...')
        try:
            path = downloader.download(model_id, progress_callback=print)
            print(f'[OK] {model_id} -> {path}')
        except Exception as e:
            print(f'[WARN] Error en {model_id}: {e}')
    else:
        print(f'[OK] {model_id} ya instalado.')

print('\nEstado final:')
print(downloader.status_report())


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA 6 — Lanzar el Dashboard MammoAI                     ║
# ╚══════════════════════════════════════════════════════════════╝
import importlib, sys, subprocess, logging
from pathlib import Path

# Logging live en consola de Jupyter/Colab
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(name)s - %(message)s',
    datefmt='%H:%M:%S',
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
print('[LOG] Logging activado - veras en tiempo real todo lo que pasa en el dashboard.')

if Path('/content/IA-Models').exists():
    subprocess.run(['git', 'fetch', 'origin', 'main'], cwd='/content/IA-Models', capture_output=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd='/content/IA-Models', capture_output=True)
    print('[OK] Repositorio actualizado desde GitHub.')

import gradio as gr
try:
    gr.close_all()
except Exception:
    pass

import app as mammo_app
importlib.reload(mammo_app)

print('Iniciando MammoAI Dashboard...')
print('   El enlace publico aparecera en unos segundos.')
print()

demo = mammo_app.build_app()
mammo_app.launch_app(demo, share=True)


---

## 🔧 Celdas de Utilidad (Opcionales)

Las siguientes celdas son herramientas adicionales que puedes ejecutar según necesites.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  UTILIDAD A - Descarga CBIS-DDSM real via TFDS (~6 GB)     ║
# ╚══════════════════════════════════════════════════════════════╝
# Dataset oficial: curated_breast_imaging_ddsm
# Ref: https://www.tensorflow.org/datasets/catalog/curated_breast_imaging_ddsm
# Req: Celda 2 debe estar ejecutada (instala tensorflow-datasets)

from tasks.breast_cancer.dataset import CBISDDSMDataset

print('[DOWNLOAD] Descargando CBIS-DDSM oficial desde TensorFlow Datasets...')
print('   Fuente: curated_breast_imaging_ddsm (certificado NIH/TCIA)')
print('   Puede tardar 15-45 minutos en Colab.')
print()

dataset = CBISDDSMDataset(
    split='train',
    max_samples=None,
)

print(f'\n[OK] Dataset listo: {len(dataset)} muestras')
print(f'   Benignas:  {sum(1 for _,l in dataset.samples if l==0)}')
print(f'   Malignas:  {sum(1 for _,l in dataset.samples if l==1)}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  UTILIDAD B — Push manual a GitHub desde Colab             ║
# ╚══════════════════════════════════════════════════════════════╝
from git_utils import GitManager

if not GITHUB_TOKEN:
    print('⚠️  Configura GITHUB_TOKEN en Colab Secrets primero.')
else:
    gm = GitManager(
        token=GITHUB_TOKEN,
        local_dir='/content/IA-Models',
    )
    status = gm.get_status()
    print('📋 Estado del repositorio:')
    for k, v in status.items():
        print(f'   {k}: {v}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  UTILIDAD C — Inferencia con logs detallados en consola    ║
# ╚══════════════════════════════════════════════════════════════╝
import importlib, logging, time, sys
from pathlib import Path
from PIL import Image
import numpy as np
import torch

# ── Configurar logs visibles en la celda ─────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s [%(levelname)s] %(name)s — %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
for noisy in ['urllib3', 'filelock', 'PIL', 'matplotlib']:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print('=' * 65)
print('  MAMMOAI — DIAGNÓSTICO CON LOGS DETALLADOS')
print('=' * 65)

# ── 1. Cargar módulos ─────────────────────────────────────────────
print('\n[1/6] Importando módulos del proyecto...')
t0 = time.time()
import tasks
from core.registry import TaskRegistry
from dicom_utils import load_image
from detector import full_diagnostic_pipeline
print(f'     OK ({time.time()-t0:.2f}s)')

# ── 2. Imagen de prueba ───────────────────────────────────────────
print('\n[2/6] Preparando imagen de entrada...')
# Cambia esta ruta a tu imagen .pgm / .png / .dcm / .jpg
TEST_IMAGE_PATH = '/content/test_mammogram.pgm'

if not Path(TEST_IMAGE_PATH).exists():
    print(f'     Imagen no encontrada. Generando PGM 16-bit sintético...')
    base = np.random.normal(120, 25, (512, 512)).clip(0, 4095).astype(np.uint16)
    cy, cx = np.random.randint(180, 330), np.random.randint(180, 330)
    for dy in range(-40, 41):
        for dx in range(-40, 41):
            if dy*dy + dx*dx <= 40*40:
                base[cy+dy, cx+dx] = min(4095, int(base[cy+dy, cx+dx]) + 1500)
    with open(TEST_IMAGE_PATH, 'wb') as f:
        f.write(b'P5\n512 512\n4095\n')
        f.write(base.astype('>u2').tobytes())
    print(f'     Imagen PGM creada: {TEST_IMAGE_PATH}')
else:
    print(f'     Usando imagen: {TEST_IMAGE_PATH}')

# ── 3. Cargar imagen ──────────────────────────────────────────────
print('\n[3/6] Cargando imagen...')
t0 = time.time()
pil_image, pixel_spacing, meta = load_image(TEST_IMAGE_PATH)
print(f'     Formato:       {meta.get("Format", "?")}')
print(f'     Dimensiones:   {pil_image.width} x {pil_image.height} px')
print(f'     Pixel spacing: {pixel_spacing:.4f} mm/px')
print(f'     Bit depth:     {meta.get("BitDepth", "N/A")}')
print(f'     OK ({time.time()-t0:.2f}s)')

# ── 4. Cargar tarea y modelo ──────────────────────────────────────
print('\n[4/6] Cargando tarea y modelo...')
t0 = time.time()
MODEL_ID = 'efficientnet_b4_cbis'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'     Modelo:  {MODEL_ID}')
print(f'     Device:  {DEVICE}')
task = TaskRegistry.load_task('breast_cancer')
print(f'     Tarea:   {task.task_name}')
print(f'     Formatos soportados: {task.supported_formats}')
print(f'     OK ({time.time()-t0:.2f}s)')

# ── 5. Pipeline de diagnóstico ────────────────────────────────────
print('\n[5/6] Ejecutando pipeline de diagnóstico...')
print('     (Primer uso descarga pesos ImageNet ~90 MB)')
t0 = time.time()
annotated, heatmap, report = full_diagnostic_pipeline(
    image=pil_image,
    model_id=MODEL_ID,
    task=task,
    pixel_spacing=pixel_spacing,
    confidence_threshold=0.30,
    generate_gradcam=True,
    device=DEVICE,
)
elapsed = time.time() - t0
print(f'     OK — inferencia en {elapsed:.2f}s')

# ── 6. Mostrar resultados ─────────────────────────────────────────
print('\n[6/6] Resultados:')
clf = report.get('classification', {})
print(f'     Clasificación:  {clf.get("predicted_class", "N/A")} ({clf.get("confidence", 0):.1f}%)')
print(f'     Probabilidades: {clf.get("probabilities", {})}')
birads_info = report.get('birads_info', {})
print(f'     BIRADS:         {birads_info.get("label", "?")} — {birads_info.get("meaning", "")}')
dets = report.get('detections', [])
print(f'     Detecciones:    {len(dets)} lesion(es)')
for d in dets:
    print(f'       #{d["id"]} {d["class"]:15s} | conf={d["confidence"]:.1f}% | diam={d["diameter_mm"]:.1f}mm')

print('\n' + '=' * 65)
print(report['report_text'])
print('=' * 65)

from IPython.display import display
print('\nImagen anotada:')
display(annotated)
print('\nMapa de calor Grad-CAM:')
display(heatmap)
